# TestFitting.ipynb — 피팅 성능 비교

**전처리 파이프라인**: IMU+PCA 정렬 → Mean Slice (0~100%, n=10) → DBSCAN 노이즈 제거

| 단계 | 내용 |
|------|------|
| 1 | **셀 [1]** 실행: 경로 설정 + 공통 함수 로드 |
| 2 | **셀 [3]** 실행: 전체 프레임 전처리 → `frames_data` 생성 |
| 3 | 이후 셀에 피팅 코드 추가 후 `frames_data` 활용 |

In [ ]:
# ============================================================================
# ★ 경로 설정 — 서버/환경에 맞게 이곳만 수정하세요
# ============================================================================
ROOT_PATH  = "/workspace/tree"              # 프로젝트 루트
LABEL_CSV  = "taehee/label_filtered_f.csv" # ROOT_PATH 기준 label 파일 경로
FRAME_IDS  = list(range(3, 52))            # 분석할 프레임 범위
# ============================================================================

import numpy as np
import json
import pandas as pd
from pathlib import Path
from sklearn.cluster import DBSCAN
from tqdm import tqdm


# ── 데이터 로더 ──────────────────────────────────────────────

class TreeDataLoader:
    def __init__(self, root_dir, label_csv=None):
        self.root_dir    = Path(root_dir)
        _lcsv            = label_csv if label_csv is not None else LABEL_CSV
        self._label_path = self.root_dir / _lcsv

    def load_frame(self, frame_id):
        pts_path = self.root_dir / 'clustered_dataset' / f'points_{frame_id:04d}.bin'
        if not pts_path.exists():
            pts_path = self.root_dir.parent / 'clustered_dataset' / f'points_{frame_id:04d}.bin'
        points = np.fromfile(pts_path, dtype=np.float32).reshape(-1, 3)

        imu_path = self.root_dir / 'raw_dataset_filtered' / f'IMU_{frame_id}.json'
        with open(imu_path) as f:
            imu = json.load(f)

        df  = pd.read_csv(self._label_path)
        row = df[df['frame'] == frame_id].iloc[0]
        return {'points': points, 'imu': imu, 'actual_dbh': float(row['dbh'])}


# ── 정렬 함수 ────────────────────────────────────────────────

def rotation_matrix_from_vectors(vec1, vec2):
    a = vec1 / (np.linalg.norm(vec1) + 1e-12)
    b = vec2 / (np.linalg.norm(vec2) + 1e-12)
    v = np.cross(a, b)
    c = np.dot(a, b)
    if abs(c + 1.0) < 1e-10:
        perp = np.array([1, 0, 0]) if abs(a[0]) < 0.9 else np.array([0, 1, 0])
        v = np.cross(a, perp)
        v /= np.linalg.norm(v) + 1e-12
        return 2 * np.outer(v, v) - np.eye(3)
    s    = np.linalg.norm(v)
    kmat = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + kmat + kmat @ kmat * ((1 - c) / (s ** 2 + 1e-12))


def align_with_imu(points, imu):
    rot      = np.array(imu['rotation_matrix'])
    aligned  = points @ rot.T
    centroid = aligned.mean(axis=0)
    return aligned - centroid, rot, centroid


def align_with_pca(points, target_up=np.array([0, 1, 0])):
    centroid   = points.mean(axis=0)
    centered   = points - centroid
    cov        = np.cov(centered.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    idx        = np.argsort(eigvals)[::-1]
    trunk_axis = eigvecs[:, idx[0]]
    if trunk_axis @ target_up < 0:
        trunk_axis = -trunk_axis
    R       = rotation_matrix_from_vectors(trunk_axis, target_up)
    aligned = centered @ R.T
    return aligned, R, centroid, trunk_axis, eigvals[idx], eigvecs[:, idx]


def combined_alignment(points, imu):
    aligned_imu, rot_imu, centroid_imu = align_with_imu(points, imu)
    aligned_pca, rot_pca, *_           = align_with_pca(aligned_imu)
    return aligned_pca, {'rot_imu': rot_imu, 'rot_pca': rot_pca,
                         'centroid_imu': centroid_imu}


# ── 슬라이스 추출 ─────────────────────────────────────────────

def extract_mean_slice(points_aligned, height_range=(0.0, 1.0),
                       n_slices=10, thickness=0.02):
    """
    IMU+PCA 정렬된 포인트에서 Mean Slice 추출 (XZ 평면)
    - height_range : Y축 기준 높이 비율 (0.0=하단, 1.0=상단)
    - n_slices     : 슬라이스 개수
    - thickness    : 슬라이스 두께 (m)
    각 슬라이스를 중심 정규화 후 XZ 포인트 전체 합산
    Returns: (n, 2) ndarray, 단위 m
    """
    y       = points_aligned[:, 1]
    y_lo    = y.min() + (y.max() - y.min()) * height_range[0]
    y_hi    = y.min() + (y.max() - y.min()) * height_range[1]
    heights = np.linspace(y_lo, y_hi, n_slices)

    all_pts = []
    for h in heights:
        mask = np.abs(y - h) < thickness / 2
        if mask.sum() > 5:
            xz  = points_aligned[mask][:, [0, 2]]
            xz -= xz.mean(axis=0)    # 슬라이스 중심 → 원점
            all_pts.append(xz)

    return np.vstack(all_pts) if all_pts else np.empty((0, 2))


def filter_noise_dbscan(xz, eps=0.05, min_samples=5):
    """DBSCAN으로 가장 큰 클러스터만 남김"""
    if len(xz) < min_samples:
        return xz, np.zeros(len(xz), dtype=int)
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit(xz).labels_
    unique = [l for l in set(labels) if l != -1]
    if not unique:
        return xz, labels
    best = max(unique, key=lambda l: np.sum(labels == l))
    return xz[labels == best], labels


print(f"ROOT_PATH  : {ROOT_PATH}")
print(f"LABEL_CSV  : {ROOT_PATH}/{LABEL_CSV}")
print(f"FRAME_IDS  : {FRAME_IDS[0]} ~ {FRAME_IDS[-1]}  ({len(FRAME_IDS)}개)")
print("공통 함수 로드 완료.")


## 데이터 전처리

**파이프라인**: `load_frame` → `combined_alignment` (IMU+PCA) → `extract_mean_slice` (0~100%, n=10) → `filter_noise_dbscan`

실행 결과 `frames_data`: 프레임별 `{'frame_id', 'actual_dbh', 'xz_clean'}` 리스트

In [ ]:
loader      = TreeDataLoader(ROOT_PATH)
frames_data = []   # 피팅에 사용할 전처리 결과
skip_count  = 0

for frame_id in tqdm(FRAME_IDS, desc='전처리'):
    try:
        data   = loader.load_frame(frame_id)
        points = data['points']
        imu    = data['imu']
        dbh    = data['actual_dbh']

        # ① IMU + PCA 정렬
        aligned, _ = combined_alignment(points, imu)

        # ② Mean Slice (0~100%, 10 슬라이스, 두께 2cm)
        xz_raw = extract_mean_slice(aligned, height_range=(0.0, 1.0),
                                    n_slices=10, thickness=0.02)
        if len(xz_raw) < 6:
            skip_count += 1
            continue

        # ③ DBSCAN 노이즈 제거
        xz_clean, _ = filter_noise_dbscan(xz_raw)
        if len(xz_clean) < 6:
            skip_count += 1
            continue

        frames_data.append({
            'frame_id'  : frame_id,
            'actual_dbh': dbh,       # cm
            'xz_clean'  : xz_clean,  # (n, 2) ndarray, 단위 m (2D 피팅용)
            'pts_3d'    : aligned,   # (n, 3) ndarray, 단위 m (3D 실린더 피팅용)
        })

    except Exception as e:
        skip_count += 1
        print(f'  [skip] frame {frame_id}: {e}')

print(f"\n전처리 완료: {len(frames_data)}개 프레임 (스킵 {skip_count}개)")
print(f"DBH 범위: {min(d['actual_dbh'] for d in frames_data):.1f} ~ "
      f"{max(d['actual_dbh'] for d in frames_data):.1f} cm")
print(f"평균 포인트 수: {np.mean([len(d['xz_clean']) for d in frames_data]):.0f}개/프레임")
print("\n→ frames_data 준비 완료. 아래 셀에 피팅 코드를 추가하세요.")


## (선택) 단일 프레임 시각화

`frames_data[idx]` 의 `xz_clean` 포인트를 플롯하여 전처리 상태를 확인합니다.

In [ ]:
import matplotlib.pyplot as plt

idx = 0                     # 확인할 프레임 인덱스
d   = frames_data[idx]
xz  = d['xz_clean']

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(xz[:, 0] * 100, xz[:, 1] * 100, s=3, alpha=0.5)
ax.set_title(f"Frame {d['frame_id']}  |  실제 DBH = {d['actual_dbh']:.1f} cm")
ax.set_xlabel("X (cm)")
ax.set_ylabel("Z (cm)")
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"포인트 수: {len(xz)}")
print(f"X 범위: {xz[:,0].min()*100:.1f} ~ {xz[:,0].max()*100:.1f} cm")
print(f"Z 범위: {xz[:,1].min()*100:.1f} ~ {xz[:,1].max()*100:.1f} cm")


## 피팅 방법 비교

| # | 방법 | 핵심 아이디어 |
|---|------|---------------|
| 0 | **기준 Circle LSQ** | 일반 최소제곱 원 피팅 (기준선) |
| N | **Pratt Algebraic** | 고유값 분해 기반 대수적 피팅 — 부분 호(Arc) 데이터에서 LSQ보다 안정적 |
| 1 | **3D RANSAC Cylinder** | 3D 원통 모델, 축 방향 동시 추정, 이상치 자동 제거 |
| 2 | **Partial Arc (Curvature)** | 외곽 경계 법선 벡터 교점으로 중심 추정 (부분 스캔 특화) |
| 3 | **Robust LSQ — Huber** | Huber 손실함수 기반 IRLS, 완만한 이상치 억제 |
| 4 | **Robust LSQ — Bisquare** | Tukey Bisquare 기반 IRLS, 강한 이상치 완전 차단 |

**전처리 옵션**: Voxel Downsampling → Statistical Outlier Removal → **DBSCAN 군집화** (나무 기둥 클러스터만 추출)

**후처리**: Kalman Filter — 프레임 시계열로 DBH 상태를 유지, 결측/이상치를 부드러운 연속 예측값으로 대체

In [ ]:
from scipy.spatial import ConvexHull, KDTree
from scipy.optimize import least_squares
from scipy.linalg import eig as scipy_eig
from sklearn.cluster import DBSCAN as _DBSCAN

# ============================================================
# 전처리: Voxel Downsampling + SOR + DBSCAN Clustering
# ============================================================

def voxel_downsample(xz, voxel_size=0.008):
    """복셀 그리드로 포인트 밀도 균일화"""
    if len(xz) < 4:
        return xz
    keys = np.floor(xz / voxel_size).astype(int)
    _, idx = np.unique(keys, axis=0, return_index=True)
    return xz[np.sort(idx)]


def statistical_outlier_removal(xz, k=10, std_ratio=2.0):
    """이웃 거리 기반 통계적 이상치 제거"""
    if len(xz) < k + 1:
        return xz
    dists, _ = KDTree(xz).query(xz, k=k + 1)
    mean_d = dists[:, 1:].mean(axis=1)
    thresh = mean_d.mean() + std_ratio * mean_d.std()
    return xz[mean_d < thresh]


def preprocess(xz, do_voxel=True, do_sor=True, do_dbscan=True,
               dbscan_eps=0.04, dbscan_min_samples=5):
    """Voxel Downsampling → SOR → DBSCAN 군집화 순서 전처리

    do_dbscan: 피팅 직전 DBSCAN으로 가장 큰 군집(나무 기둥)만 추출
               숲 노이즈/지면 포인트 혼입으로 원이 찌그러지는 현상 방지
    """
    if do_voxel:
        xz = voxel_downsample(xz)
    if do_sor:
        xz = statistical_outlier_removal(xz)
    if do_dbscan and len(xz) >= dbscan_min_samples:
        labels = _DBSCAN(eps=dbscan_eps, min_samples=dbscan_min_samples).fit(xz).labels_
        unique = [l for l in set(labels) if l != -1]
        if unique:
            best = max(unique, key=lambda l: np.sum(labels == l))
            xz = xz[labels == best]
    return xz


# ============================================================
# 기준: Circle LSQ (Geometric Fitting)
# ============================================================

def fit_circle_lsq(xz):
    """일반 최소제곱 원 피팅 → diameter (cm) or None"""
    if len(xz) < 5:
        return None
    def residuals(params):
        cx, cy, r = params
        return np.sqrt((xz[:,0]-cx)**2 + (xz[:,1]-cy)**2) - r
    x0 = [xz[:,0].mean(), xz[:,1].mean(),
           np.std(np.linalg.norm(xz - xz.mean(axis=0), axis=1))]
    try:
        res = least_squares(residuals, x0)
        r = abs(res.x[2])
        return r * 2 * 100 if 0.05 <= r <= 0.5 else None
    except Exception:
        return None


# ============================================================
# New: Pratt Algebraic Circle Fitting
# ============================================================

def fit_circle_pratt(xz):
    """
    Pratt 대수적 원 피팅 (Algebraic Fitting)

    일반 LSQ(Geometric)는 부분 호(Arc) 데이터에서 반지름이 발산하는 경향.
    Pratt은 고유값 분해로 원을 구하여 편향된 데이터에서 훨씬 안정적.

    최소화: sum( a(x²+y²) + bx + cy + d )²
    제약:   b² + c² - 4ad = 1  (원이 타원으로 찌그러짐 방지)
    → 일반화 고유값 문제: M·v = λ·B·v 의 최솟값 양수 고유벡터 선택

    Returns: diameter (cm) or None
    """
    if len(xz) < 5:
        return None
    x, y = xz[:, 0].copy(), xz[:, 1].copy()
    # 수치 안정성: 중심화
    x -= x.mean()
    y -= y.mean()

    z = x**2 + y**2
    Z = np.column_stack([z, x, y, np.ones(len(x))])
    M = Z.T @ Z

    # Pratt 제약 행렬 (계수 [a, b, c, d] 기준)
    B_pratt = np.array([
        [ 0,  0,  0, -2],
        [ 0,  1,  0,  0],
        [ 0,  0,  1,  0],
        [-2,  0,  0,  0],
    ], dtype=float)

    try:
        eigvals, eigvecs = scipy_eig(M, B_pratt)
    except Exception:
        return None

    # 실수이고 양수인 고유값 중 최솟값 선택
    real_mask = (np.abs(eigvals.imag) < 1e-8) & (eigvals.real > 1e-10)
    if not real_mask.any():
        return None
    real_idxs = np.where(real_mask)[0]
    min_idx   = real_idxs[np.argmin(eigvals.real[real_mask])]
    v         = eigvecs[:, min_idx].real

    a, b, c, d = v
    if abs(a) < 1e-10:
        return None
    cx   = -b / (2 * a)
    cy   = -c / (2 * a)
    r_sq = (b**2 + c**2 - 4 * a * d) / (4 * a**2)
    if r_sq <= 0:
        return None
    r = np.sqrt(r_sq)
    return r * 2 * 100 if 0.05 <= r <= 0.5 else None


# ============================================================
# 1. 3D RANSAC Cylinder Fitting
# ============================================================

def fit_cylinder_3d_ransac(pts_3d, height_range=(0.3, 0.7),
                           n_iter=3000, inlier_thresh=0.015):
    """
    3D RANSAC 원통 피팅
    - 3개 포인트로 단면 평면 결정 → 평면 법선 = 원통 축
    - 평면에 투영된 포인트로 원 피팅
    - 3D 원통 거리로 인라이어 계산 (기울기 보정 내장)
    Returns: diameter (cm) or None
    """
    y = pts_3d[:, 1]
    lo  = y.min() + (y.max() - y.min()) * height_range[0]
    hi  = y.min() + (y.max() - y.min()) * height_range[1]
    pts = pts_3d[(y >= lo) & (y <= hi)]
    if len(pts) < 10:
        return None

    rng = np.random.default_rng(42)
    best_r, best_n = 0.0, 0
    best_axis, best_axis_pt = np.array([0.,1.,0.]), pts.mean(axis=0)

    for _ in range(n_iter):
        i0, i1, i2 = rng.choice(len(pts), 3, replace=False)
        p1, p2, p3 = pts[i0], pts[i1], pts[i2]

        axis = np.cross(p2 - p1, p3 - p1)
        a_n  = np.linalg.norm(axis)
        if a_n < 1e-10:
            continue
        axis = axis / a_n
        if axis[1] < 0:
            axis = -axis
        if axis[1] < 0.5:
            continue

        e1 = np.array([1.,0.,0.]) - axis[0] * axis
        e1_n = np.linalg.norm(e1)
        if e1_n < 1e-10:
            continue
        e1 /= e1_n
        e2 = np.cross(axis, e1)

        def to2d(p):
            d = p - p1
            return np.array([np.dot(d, e1), np.dot(d, e2)])
        q1, q2, q3 = to2d(p1), to2d(p2), to2d(p3)

        ax,ay = q1; bx,by = q2; cx,cy = q3
        d = 2*(ax*(by-cy)+bx*(cy-ay)+cx*(ay-by))
        if abs(d) < 1e-10:
            continue
        ux = ((ax**2+ay**2)*(by-cy)+(bx**2+by**2)*(cy-ay)+(cx**2+cy**2)*(ay-by))/d
        uy = ((ax**2+ay**2)*(cx-bx)+(bx**2+by**2)*(ax-cx)+(cx**2+cy**2)*(bx-ax))/d
        r  = np.sqrt((ax-ux)**2+(ay-uy)**2)
        if not (0.05 <= r <= 0.5):
            continue

        axis_pt   = p1 + ux*e1 + uy*e2
        dv        = pts - axis_pt
        d_par     = np.outer(dv @ axis, axis)
        d_perp    = np.linalg.norm(dv - d_par, axis=1)
        n_in      = int(np.sum(np.abs(d_perp - r) < inlier_thresh))

        if n_in > best_n:
            best_n, best_r = n_in, r
            best_axis, best_axis_pt = axis.copy(), axis_pt.copy()

    if best_n < 5:
        return None

    dv     = pts - best_axis_pt
    d_par  = np.outer(dv @ best_axis, best_axis)
    d_perp = np.linalg.norm(dv - d_par, axis=1)
    mask   = np.abs(d_perp - best_r) < inlier_thresh
    if mask.sum() >= 5:
        e1b = np.array([1.,0.,0.]) - best_axis[0]*best_axis
        e1b /= np.linalg.norm(e1b) + 1e-12
        e2b = np.cross(best_axis, e1b)
        inl = pts[mask]
        p2d = np.column_stack([inl @ e1b, inl @ e2b])
        r_ref = fit_circle_lsq(p2d)
        if r_ref is not None:
            return r_ref

    return best_r * 2 * 100


# ============================================================
# 2. Curvature-based Partial Arc Fitting
# ============================================================

def fit_partial_arc_curvature(xz):
    """
    곡률 기반 부분 호 피팅
    - Convex Hull 외곽 경계 추출
    - 경계의 각 점에서 접선 → 법선 벡터 계산
    - 모든 법선의 교점(중심)을 최소제곱으로 추정
    - 중심까지 평균 거리 = 반지름
    Returns: diameter (cm) or None
    """
    if len(xz) < 8:
        return None
    try:
        hull     = ConvexHull(xz)
        centroid = xz.mean(axis=0)

        verts  = xz[hull.vertices]
        angles = np.arctan2(verts[:,1]-centroid[1], verts[:,0]-centroid[0])
        verts  = verts[np.argsort(angles)]
        n      = len(verts)
        if n < 5:
            return None

        normals, pts_n = [], []
        for i in range(1, n - 1):
            tan = verts[(i+1)%n] - verts[(i-1)%n]
            t_n = np.linalg.norm(tan)
            if t_n < 1e-10:
                continue
            tan /= t_n
            normal = np.array([-tan[1], tan[0]])
            if np.dot(normal, centroid - verts[i]) < 0:
                normal = -normal
            normals.append(normal)
            pts_n.append(verts[i])

        if len(normals) < 4:
            return None

        normals = np.array(normals)
        pts_n   = np.array(pts_n)

        A = np.column_stack([-normals[:,1], normals[:,0]])
        b = normals[:,0]*pts_n[:,1] - normals[:,1]*pts_n[:,0]
        center, _, _, _ = np.linalg.lstsq(A, b, rcond=None)

        r = float(np.mean(np.linalg.norm(pts_n - center, axis=1)))
        return r * 2 * 100 if 0.05 <= r <= 0.5 else None

    except Exception:
        return None


# ============================================================
# 3. Robust LSQ (IRLS — Huber / Bisquare)
# ============================================================

def fit_circle_robust(xz, loss='huber', max_iter=30, tol=1e-6):
    """
    강인한 최소제곱 원 피팅 (IRLS)
    loss='huber'    : Huber 손실 (완만한 이상치 억제)
    loss='bisquare' : Tukey Bisquare (강한 이상치 완전 차단)
    Returns: diameter (cm) or None
    """
    if len(xz) < 5:
        return None

    x, y = xz[:,0], xz[:,1]
    A_mat = np.column_stack([x, y, np.ones(len(x))])
    b_vec = -(x**2 + y**2)

    try:
        c, _, _, _ = np.linalg.lstsq(A_mat, b_vec, rcond=None)
    except Exception:
        return None
    cx, cy = -c[0]/2, -c[1]/2
    r_sq   = cx**2 + cy**2 - c[2]
    if r_sq <= 0:
        return None
    r = np.sqrt(r_sq)

    for _ in range(max_iter):
        resid = np.sqrt((x-cx)**2 + (y-cy)**2) - r
        sigma = max(np.median(np.abs(resid)) / 0.6745, 1e-10)

        if loss == 'huber':
            k = 1.345 * sigma
            w = np.where(np.abs(resid) <= k, 1.0, k / (np.abs(resid) + 1e-12))
        else:
            k = 4.685 * sigma
            m = np.abs(resid) <= k
            w = np.where(m, (1-(resid/k)**2)**2, 0.0)

        W = np.diag(w)
        try:
            c_new, _, _, _ = np.linalg.lstsq(A_mat.T @ W @ A_mat,
                                              A_mat.T @ (W @ b_vec), rcond=None)
        except Exception:
            break

        cx_n, cy_n = -c_new[0]/2, -c_new[1]/2
        r_sq_n = cx_n**2 + cy_n**2 - c_new[2]
        if r_sq_n <= 0:
            break
        r_new = np.sqrt(r_sq_n)

        if abs(r_new - r) < tol:
            break
        cx, cy, r = cx_n, cy_n, r_new

    return r * 2 * 100 if 0.05 <= r <= 0.5 else None


print('피팅 함수 로드 완료.')
print('  fit_circle_lsq            — 기준: Circle LSQ (Geometric)')
print('  fit_circle_pratt          — New: Pratt Algebraic (부분 호 특화)')
print('  fit_cylinder_3d_ransac    — 1: 3D RANSAC Cylinder')
print('  fit_partial_arc_curvature — 2: Partial Arc (Curvature)')
print('  fit_circle_robust(huber)  — 3: Robust LSQ Huber')
print('  fit_circle_robust(bisq.)  — 4: Robust LSQ Bisquare')
print('  preprocess(voxel+SOR+DBSCAN) — 전처리 옵션 포함')

## 오차율 비교 실행

모든 피팅 방법 × 전처리 유무를 `frames_data` 전체 프레임에 실행하고
**RMSE / MAE / Bias** 를 비교합니다.


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── 이상 예측값 필터 설정 ─────────────────────────────────────
OUTLIER_ABS_CM  = 15.0   # |pred - actual| > 이 값(cm) → 결측
OUTLIER_RATIO   = 0.5    # |pred/actual - 1| > 이 값(비율) → 결측

def reject_outlier(pred, actual):
    """이상 예측값 → None, 정상값은 그대로 반환"""
    if pred is None:
        return None
    if abs(pred - actual) > OUTLIER_ABS_CM:
        return None
    if abs(pred / actual - 1.0) > OUTLIER_RATIO:
        return None
    return pred


# ── Kalman Filter: DBH 시간적 평활화 ─────────────────────────
KALMAN_Q = 1.0   # process noise (cm²) — 프레임간 DBH 변화 분산
KALMAN_R = 9.0   # measurement noise (cm²) — 피팅 불확실성

class KalmanDBH:
    """1D Kalman Filter — 나무 DBH 시간적 평활화

    나무는 고정된 물체이므로 이전 프레임 결과를 활용합니다.
    - measurement=None : predict-only (불확실성 증가, 이전 추정값 유지)
    - measurement=값   : Kalman gain으로 가중 업데이트
    결과: 결측/이상 프레임도 '결측' 대신 부드러운 연속 추정값으로 채워짐
    """
    def __init__(self, q=KALMAN_Q, r=KALMAN_R):
        self.q = q
        self.r = r
        self.x = None    # 상태 추정값 (DBH, cm)
        self.P = 100.0   # 오차 공분산 (초기: 큰 불확실성)

    def update(self, measurement):
        # Predict: 시간 경과 → 불확실성 증가
        if self.x is not None:
            self.P += self.q

        if measurement is None:
            return self.x      # 측정값 없으면 현재 추정값만 반환

        if self.x is None:
            # 첫 유효 측정값으로 초기화
            self.x = measurement
            self.P = self.r
            return self.x

        # Update: Kalman gain으로 가중 평균
        K      = self.P / (self.P + self.r)
        self.x = self.x + K * (measurement - self.x)
        self.P = (1 - K) * self.P
        return self.x


# ── 피팅 방법 정의 ─────────────────────────────────────────────
METHODS = [
    ('circle_lsq',      '기준 Circle LSQ',          lambda xz, d: fit_circle_lsq(xz)),
    ('pratt',           'Pratt Algebraic',           lambda xz, d: fit_circle_pratt(xz)),
    ('ransac_3d',       '3D RANSAC Cylinder',        lambda xz, d: fit_cylinder_3d_ransac(d['pts_3d'])),
    ('arc_curvature',   'Partial Arc (Curvature)',   lambda xz, d: fit_partial_arc_curvature(xz)),
    ('robust_huber',    'Robust LSQ — Huber',        lambda xz, d: fit_circle_robust(xz, 'huber')),
    ('robust_bisquare', 'Robust LSQ — Bisquare',     lambda xz, d: fit_circle_robust(xz, 'bisquare')),
]

# ── 피팅 실행: 원본 / 전처리(voxel+SOR+DBSCAN) ────────────────
rows = []
for d in tqdm(frames_data, desc='피팅'):
    xz_raw  = d['xz_clean']
    xz_prep = preprocess(xz_raw)     # Voxel + SOR + DBSCAN
    actual  = d['actual_dbh']
    row     = {'frame_id': d['frame_id'], 'actual_dbh': actual}

    for key, _, fn in METHODS:
        for suffix, xz_use in [('', xz_raw), ('_prep', xz_prep)]:
            try:
                # ransac_3d 는 lambda 내부에서 pts_3d 사용 → xz_use 무관
                raw_v = fn(xz_use, d)
                v     = reject_outlier(raw_v, actual)
                row[f'pred_{key}{suffix}']  = v
                row[f'error_{key}{suffix}'] = (v - actual) if v is not None else None
            except Exception:
                row[f'pred_{key}{suffix}']  = None
                row[f'error_{key}{suffix}'] = None

    rows.append(row)

df_fit = pd.DataFrame(rows)

# ── Kalman Filter 적용 (전처리 결과 기준) ─────────────────────
# 프레임 순서(frame_id 순)로 각 방법의 예측을 Kalman으로 평활화
for key, _, _ in METHODS:
    pred_col   = f'pred_{key}_prep'
    kalman_col = f'kalman_{key}'
    if pred_col not in df_fit.columns:
        continue
    kf = KalmanDBH()
    kv = []
    for v in df_fit[pred_col]:
        meas = float(v) if (v is not None and not pd.isna(v)) else None
        kv.append(kf.update(meas))
    df_fit[kalman_col]            = kv
    df_fit[f'kalman_error_{key}'] = df_fit.apply(
        lambda r: (r[kalman_col] - r['actual_dbh'])
                  if (r[kalman_col] is not None and not pd.isna(r[kalman_col])) else None,
        axis=1)

# ── 결과 표 출력 ─────────────────────────────────────────────
print("\n" + "="*82)
print(f"{'방법':<34} {'RMSE':>7} {'MAE':>7} {'Bias':>9} {'n':>5}  모드")
print('='*82)

summary_rows = []
for key, label, _ in METHODS:
    configs = [
        (f'error_{key}',        '원본',         f'pred_{key}'),
        (f'error_{key}_prep',   '전처리',        f'pred_{key}_prep'),
        (f'kalman_error_{key}', 'Kalman(prep)',  f'kalman_{key}'),
    ]
    for err_col, mode_label, _ in configs:
        if err_col not in df_fit.columns:
            continue
        v = pd.to_numeric(df_fit[err_col], errors='coerce').dropna()
        if len(v) == 0:
            continue
        rmse = float(np.sqrt(np.mean(v**2)))
        mae  = float(np.mean(np.abs(v)))
        bias = float(np.mean(v))
        print(f'{label:<34} {rmse:>7.2f} {mae:>7.2f} {bias:>+9.2f} {len(v):>5}  {mode_label}')
        summary_rows.append({'method': label, 'mode': mode_label,
                             'rmse': rmse, 'mae': mae, 'bias': bias, 'n': len(v)})
    print()

print('='*82)
summary_df = pd.DataFrame(summary_rows)
best = summary_df.loc[summary_df['rmse'].idxmin()]
print(f"\n★ 최저 RMSE: [{best['mode']}] {best['method']}  →  {best['rmse']:.2f} cm")

# ── 시각화 ───────────────────────────────────────────────────
from pathlib import Path
out_dir = Path('fitting_results')
out_dir.mkdir(exist_ok=True)

MODE_COLORS = {
    '원본':          'rgba(72,120,207,1.0)',
    '전처리':        'rgba(214,95,95,1.0)',
    'Kalman(prep)':  'rgba(46,172,89,1.0)',
}
MODE_LIGHT = {
    '원본':          'rgba(72,120,207,0.4)',
    '전처리':        'rgba(214,95,95,0.4)',
    'Kalman(prep)':  'rgba(46,172,89,0.4)',
}

# ① Bar chart: RMSE + MAE 비교 (원본/전처리/Kalman)
labels_all   = [f"{r['method']}\n({r['mode']})" for _, r in summary_df.iterrows()]
solid_colors = [MODE_COLORS[r['mode']] for _, r in summary_df.iterrows()]
light_colors = [MODE_LIGHT[r['mode']]  for _, r in summary_df.iterrows()]

fig = go.Figure([
    go.Bar(name='RMSE', x=labels_all, y=summary_df['rmse'].tolist(),
           marker_color=solid_colors,
           text=[f"{v:.2f}" for v in summary_df['rmse']], textposition='outside'),
    go.Bar(name='MAE',  x=labels_all, y=summary_df['mae'].tolist(),
           marker_color=light_colors,
           text=[f"{v:.2f}" for v in summary_df['mae']],  textposition='outside'),
])
fig.update_layout(
    title='<b>피팅 방법별 오차 비교 (파랑=원본, 빨강=전처리, 초록=Kalman)</b>',
    xaxis_title='방법', yaxis_title='오차 (cm)',
    barmode='group', width=1500, height=580,
)
fig.write_html(out_dir / 'error_bar.html')
fig.show()

# ② Time-series: 최저 RMSE 방법의 프레임별 예측 vs 실측 + Kalman
kalman_summary = summary_df[summary_df['mode'] == 'Kalman(prep)']
if len(kalman_summary) > 0:
    best_label  = kalman_summary.loc[kalman_summary['rmse'].idxmin(), 'method']
    best_key_id = next((k for k, lb, _ in METHODS if lb == best_label), METHODS[0][0])
else:
    best_label, best_key_id = METHODS[0][1], METHODS[0][0]

frame_ids = df_fit['frame_id'].tolist()
actual_v  = df_fit['actual_dbh'].tolist()
raw_v     = [float(v) if (v is not None and not pd.isna(v)) else None
             for v in df_fit[f'pred_{best_key_id}_prep']]
kalman_v  = [float(v) if (v is not None and not pd.isna(v)) else None
             for v in df_fit[f'kalman_{best_key_id}']]

fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=frame_ids, y=actual_v, mode='lines+markers',
                          name='실측 DBH',
                          line=dict(color='black', width=2),
                          marker=dict(size=5)))
fig3.add_trace(go.Scatter(x=frame_ids, y=raw_v, mode='markers',
                          name=f'{best_label} (전처리)',
                          marker=dict(color='rgba(214,95,95,0.7)', size=7)))
fig3.add_trace(go.Scatter(x=frame_ids, y=kalman_v, mode='lines+markers',
                          name=f'{best_label} + Kalman',
                          line=dict(color='rgba(46,172,89,1.0)', width=2),
                          marker=dict(size=5)))
fig3.update_layout(
    title=f'<b>프레임별 DBH 예측 — {best_label} (전처리 vs Kalman)</b>',
    xaxis_title='Frame ID', yaxis_title='DBH (cm)',
    width=1100, height=460,
)
fig3.write_html(out_dir / 'timeseries.html')
fig3.show()

# ③ Scatter: Kalman 예측 vs 실측
kalman_methods = [(key, label) for key, label, _ in METHODS
                  if f'kalman_{key}' in df_fit.columns]
fig2 = make_subplots(1, len(kalman_methods),
                     subplot_titles=[lb for _, lb in kalman_methods])
for ci, (key, label) in enumerate(kalman_methods, 1):
    col   = f'kalman_{key}'
    yv    = pd.to_numeric(df_fit[col], errors='coerce')
    mask  = yv.notna()
    xv    = df_fit.loc[mask, 'actual_dbh']
    yv    = yv[mask]
    if len(xv) == 0:
        continue
    lo, hi = min(xv.min(), yv.min())*0.9, max(xv.max(), yv.max())*1.1
    fig2.add_trace(go.Scatter(x=xv, y=yv, mode='markers',
                              marker=dict(size=7, color='rgba(46,172,89,0.8)'),
                              name=label), row=1, col=ci)
    fig2.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode='lines',
                              line=dict(dash='dash', color='gray'),
                              showlegend=False), row=1, col=ci)
    fig2.update_xaxes(title_text='실제 DBH (cm)', row=1, col=ci)
    fig2.update_yaxes(title_text='예측 DBH (cm)', row=1, col=ci)
fig2.update_layout(title='<b>예측 vs 실제 DBH — Kalman 적용</b>',
                   width=1600, height=450)
fig2.write_html(out_dir / 'pred_vs_actual_kalman.html')
fig2.show()

# CSV 저장
df_fit.to_csv(out_dir / 'results.csv', index=False)
summary_df.to_csv(out_dir / 'summary.csv', index=False)
print(f'\n결과 저장: {out_dir}/')